# Изучение развития игровой индустрии с 2000 по 2013 год

Команда игры хочет привлечь новую аудиторию и подготовить статью о развитии индустрии игр в начале XXI века. В статье-исследовании хотят сделать обзор игровых платформ, изучить объёмы продаж игр разных жанров и региональные предпочтения игроков. 

### Цели и задачи проекта

В данном проекте используются исторические данные, собранные из открытых источников, которые содержат информацию о продажах игр, сделанных в разных жанрах и выпущенных на разных платформах, а также пользовательские и экспертные оценки игр.  

**Цель проекта** — изучить историю продаж игр в начале XXI века за период с 2000 по 2013 год, провести категоризацию данных по оценкам и выделить топ-7 платформ по количеству игр, выпущенных за весь требуемый период.  

В рамках данной цели будут выполнены следующие **задачи**: загрузка и знакомство с данными, проверка ошибок в данных и их предобработка, фильтрация и категоризация данных.

### Описание данных

Используются данные датасета `new_games.csv`:
- `Name` — название игры;
- `Platform` — название платформы;
- `Year of Release` — год выпуска игры;
- `Genre` — жанр игры;
- `NA sales` — продажи в Северной Америке (в миллионах проданных копий);
- `EU sales` — продажи в Европе (в миллионах проданных копий);
- `JP sales` — продажи в Японии (в миллионах проданных копий);
- `Other sales` — продажи в других странах (в миллионах проданных копий);
- `Critic Score` — оценка критиков (от 0 до 100);
- `User Score` — оценка пользователей (от 0 до 10);
- `Rating` — рейтинг организации ESRB.

### Содержимое проекта

1. Загрузка данных и знакомство с ними  
2.  Проверка ошибок в данных и их предобработка  
3. Фильтрация данных  
4. Категоризация данных  
5. Итоговый вывод

---

## 1. Загрузка данных и знакомство с ними

In [116]:
import pandas as pd

In [117]:
df = pd.read_csv('new_games.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


In [118]:
df.head()

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


Датасет `new_games.csv` содержит 11 столбцов и 16956 строк.
Изучим типы данных и их корректность:
- **Числовые значения с плавающей запятой (float64).** 4 столбца имеют тип данных `float64`:
    - `NA sales` и `Other sales` содержат информацию о продажах, тип `float64` подходит для этих данных;
    - `Critic Score` содержит информацию об оценках критиков в диапазоне от 0 до 100, для этих данных больше подходит тип на `int8`. Оценки целочисленные, но данные содержат пропуски NaN, в связи с чем установлен тип `float64`;
    - `Year of Release` содержит сведения о годе выпуска, и для него больше подходит тип `int16`. Тип `float64` установлен по причине пропусков NaN.
- **Строковые данные (object).** 7 столбцов имеют тип данных `object`:
    - `Name`, `Platform`, `Genre` и `Rating` содержат строковую информацию (название игры, название платформы, жанр игры и рейтинг организации ESRB), здесь тип данных `object` подходит;
    - `EU sales`, `JP sales` и `User Score` хранят информацию о продажах и оценках пользователей, здесь больше подошел бы тип `float64`.

В данных содержатся пропуски в 6 столбцах: `Name`, `Year of Release`, `Genre`, `Critic Score`, `User Score` и `Rating`.

Названия столбцов корректны, отражают содержимое данных и прописаны. Для удобства работы названия столбцов можно привести к стилю snake case.



## 2.  Проверка ошибок в данных и их предобработка


### 2.1. Названия столбцов датафрейма

Приведем все столбцы к стилю snake case.

In [119]:
df.columns

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')

In [120]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

In [121]:
df.columns

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='object')

### 2.2. Типы данных

Проведем преобразование типов данных.

In [122]:
df['eu_sales'] = df['eu_sales'].replace({'unknown': None})
df['jp_sales'] = df['jp_sales'].replace({'unknown': None})
df['user_score'] = df['user_score'].replace({'tbd': None})

In [123]:
for column in ['eu_sales', 'jp_sales', 'user_score', 'na_sales', 'other_sales', 'critic_score']:
    df[column] = pd.to_numeric(df[column], downcast='float')

In [124]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  object 
 1   platform         16956 non-null  object 
 2   year_of_release  16681 non-null  float64
 3   genre            16954 non-null  object 
 4   na_sales         16956 non-null  float32
 5   eu_sales         16950 non-null  float32
 6   jp_sales         16952 non-null  float32
 7   other_sales      16956 non-null  float32
 8   critic_score     8242 non-null   float32
 9   user_score       7688 non-null   float32
 10  rating           10085 non-null  object 
dtypes: float32(6), float64(1), object(4)
memory usage: 1.0+ MB


### 2.3. Наличие пропусков в данных

- Посчитаем количество пропусков в каждом столбце в абсолютных и относительных значениях.

In [125]:
# Выводим количество пропущенных строк в датафрейме
df.isna().sum()

name                  2
platform              0
year_of_release     275
genre                 2
na_sales              0
eu_sales              6
jp_sales              4
other_sales           0
critic_score       8714
user_score         9268
rating             6871
dtype: int64

In [126]:
# Подсчитаем процент строк с пропусками
df.isna().sum() / len(df) * 100

name                0.011795
platform            0.000000
year_of_release     1.621845
genre               0.011795
na_sales            0.000000
eu_sales            0.035386
jp_sales            0.023590
other_sales         0.000000
critic_score       51.391838
user_score         54.659118
rating             40.522529
dtype: float64

В данных наблюдаются пропуски в следующих столбцах:
- `name`:  в 2 строках (0.01% данных) отсутствует информация о названии игры. Обе игры 1993 года выпуска, что не повлияет на дальнейший анализ. Данные строки можно удалить.
- `year_of_release`: в 275 строках (1.62% данных) отсутствует информация о годе выпуска игры. Так как отсутствует способ однозначного определения года выпуска игры на основе имеющихся данных, данные строки нужно удалить.
- `genre`: в 2 строках (0.01% данных) отсутствует информация о жанре. Но так как эти строки относятся к 1993 году выпуска и совпадают со строками, содержамим пропуски в названии игры, они не повлияют на ислледование. Они будут также удалены.
- `eu_sales`: в 6 строках (0.04% данных) отсутствует информация о продажах в Европе. Можно предположить, что данные игры в Европе не продавались, или данные об их продажах в Европе не были собраны. Пропуски можно заменить на на среднее значение в зависимости от названия платформы и года выхода игры.
- `jp_sales`: в 4 строках (0.02% данных) отсутствует информация о продажах в Японии. Можно предположить, что данные игры в Японии не продавались, или данные об их продажах в Японии не были собраны. Пропуски можно заменить на на среднее значение в зависимости от названия платформы и года выхода игры.
- `critic_score`: в 8714 строках (51.39% данных) отсутствует информация об оценках критиков. Доля таких строк слишком высокая, их удаление повлияет на результаты исследования. При категоризации по оценкам необходимо будет учитывать только строки с заполненными оценками. Пропуски игнорируем.
- `user_score`: в 9268 строках (54.66% данных) отсутствует информация об оценках пользователей. Доля таких строк слишком высокая, их удаление повлияет на результаты исследования. При категоризации по оценкам необходимо будет учитывать только строки с заполненными оценками. Пропуски игнорируем.
- `rating`: в 6871 строках (40.52% данных) отсутствует информация о рейтингах ESRB. Доля таких строк слишком высокая, их удаление повлияет на результаты исследования. Данный рейтинг в дальнейшем анализе не учитывается, поэтому пропуски игнорируем.

---

- Обработаем пропущенные значения.

In [127]:
# Подсчитаем количество строк в исходном датафрейме
init_row_count = df.shape[0]
init_row_count

16956

In [128]:
# Удалим все строки с пропусками в столбцах name, year_of_release, genre
df = df.dropna(subset=['name', 'year_of_release', 'genre'])

In [129]:
# Приведем тип столбца year_of_release к int16  
df['year_of_release'] = pd.to_numeric(df['year_of_release'], downcast='integer')

In [130]:
# Заполним пропуски по продажам в Европе средним значеними по платформе и году выхода игры
def mean_value_eu(row):
    if pd.isna(row['eu_sales']):
        group = df[(df['platform'] == row['platform']) & (df['year_of_release'] == row['year_of_release'])]
        return group['eu_sales'].mean()
    else:
        return row['eu_sales']
df['eu_sales'] = df.apply(mean_value_eu, axis=1)

In [131]:
# Заполним пропуски по продажам в Японии средним значеними по платформе и году выхода игры
def mean_value_jp(row):
    if pd.isna(row['jp_sales']):
        group = df[(df['platform'] == row['platform']) & (df['year_of_release'] == row['year_of_release'])]
        return group['jp_sales'].mean()
    else:
        return row['jp_sales']
df['jp_sales'] = df.apply(mean_value_jp, axis=1)

In [132]:
df.isna().sum() / len(df) * 100

name                0.000000
platform            0.000000
year_of_release     0.000000
genre               0.000000
na_sales            0.000000
eu_sales            0.000000
jp_sales            0.000000
other_sales         0.000000
critic_score       51.525871
user_score         54.685533
rating             40.637928
dtype: float64

In [133]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16679 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16679 non-null  object 
 1   platform         16679 non-null  object 
 2   year_of_release  16679 non-null  int16  
 3   genre            16679 non-null  object 
 4   na_sales         16679 non-null  float32
 5   eu_sales         16679 non-null  float64
 6   jp_sales         16679 non-null  float64
 7   other_sales      16679 non-null  float32
 8   critic_score     8085 non-null   float32
 9   user_score       7558 non-null   float32
 10  rating           9901 non-null   object 
dtypes: float32(4), float64(2), int16(1), object(4)
memory usage: 1.2+ MB


### 2.4. Явные и неявные дубликаты в данных

- Изучим уникальные значения в категориальных данных (с названиями жанра игры, платформы, рейтинга и года выпуска). Проверим, встречаются ли среди данных неявные дубликаты, связанные с опечатками или разным способом написания. Проведем нормализацию данных с текстовыми значениями.

In [134]:
df['platform'].sort_values().unique()

array(['2600', '3DO', '3DS', 'DC', 'DS', 'GB', 'GBA', 'GC', 'GEN', 'GG',
       'N64', 'NES', 'NG', 'PC', 'PCFX', 'PS', 'PS2', 'PS3', 'PS4', 'PSP',
       'PSV', 'SAT', 'SCD', 'SNES', 'TG16', 'WS', 'Wii', 'WiiU', 'X360',
       'XB', 'XOne'], dtype=object)

Данные в столбце `platform` не содержат дубликатов

In [135]:
df['genre'].sort_values().unique()

array(['ACTION', 'ADVENTURE', 'Action', 'Adventure', 'FIGHTING',
       'Fighting', 'MISC', 'Misc', 'PLATFORM', 'PUZZLE', 'Platform',
       'Puzzle', 'RACING', 'ROLE-PLAYING', 'Racing', 'Role-Playing',
       'SHOOTER', 'SIMULATION', 'SPORTS', 'STRATEGY', 'Shooter',
       'Simulation', 'Sports', 'Strategy'], dtype=object)

В столбце `genre` содержатся дубликаты, связанные с написанием жанра в разных регистрах. Приведем все названия жанров к верхнему регистру

In [136]:
df['genre'] = df['genre'].str.upper()
df['genre'].sort_values().unique()

array(['ACTION', 'ADVENTURE', 'FIGHTING', 'MISC', 'PLATFORM', 'PUZZLE',
       'RACING', 'ROLE-PLAYING', 'SHOOTER', 'SIMULATION', 'SPORTS',
       'STRATEGY'], dtype=object)

Теперь столбец `genre` дубликатов не содержит

In [137]:
df['year_of_release'].sort_values().unique()

array([1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990,
       1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001,
       2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012,
       2013, 2014, 2015, 2016], dtype=int16)

Данные в столбце `year_of_release` не содержат дубликатов

In [138]:
df['rating'].sort_values().unique()

array(['AO', 'E', 'E10+', 'EC', 'K-A', 'M', 'RP', 'T', nan], dtype=object)

Данные в столбце `rating` не содержат дубликатов. Рейтинг `K-A` является устаревшим, с 1998 года заменен на `E`. Записи за период с 2000 по 2013 годы не содержат значений с таким рейтингом.

In [139]:
# Удалим лишние пробелы в начале и конце названия игры и приведем наименование к нижнему регистру
df['name'] = df['name'].str.strip().str.lower()

- Проверим наличие явных дубликатов в данных.

In [140]:
df = df.sort_values(by=df.columns.tolist())
df.duplicated().sum()

235

In [141]:
df = df.drop_duplicates()

In [142]:
# Подсчитаем число строк в датафрейме после удаления неявных и явных дубликатов
fin_row_count = df.shape[0]
fin_row_count

16444

После нормализации данных с текстовыми значениями были выявлены 235 явных дубликата. Данные строки были удалены.

- Посчитаем количество удалённых строк в абсолютном и относительном значениях.

In [143]:
# Количество удаленных строк
del_row_count = init_row_count - fin_row_count
del_row_count

512

In [144]:
# Процент удаленных строк
round(del_row_count / init_row_count * 100, 2)

3.02

Были загружены данные `new_games.csv`. Они содержат 11 столбцов и 16956 строк, в которых содержится информация о продажах игр, сделанных в разных жанрах и выпущенных на разных платформах, а также пользовательские и экспертные оценки игр. Названия столбцов были приведены к формату snake case. При первичном знакомстве с данными и их предобработке получили такие результаты:
- В 8 столбцах (`name`, `year_of_release`, `genre`, `eu_sales`, `jp_sales`, `critic_score`, `user_score`, `rating`) были обнаружены пропущенные значения. Максимальные значения пропущенных данных — в столбцах `user_score` (54.66%), `critic_score` (51.39%), `rating` (40.52%). 
- Строки с пропущенными значениями в столбцах `name` и `genre` были удалены, так как они содержали информацию об играх 1993 года выпуска, что не повлияет на дальнейших анализ за период с 2000 по 2013 годы. Также были удалены строки с пропущенными значениями в столбце `year_of_release`, так как отсутствует способ однозначного определения года выпуска игры на основе имеющихся данных, а данный столбец необходим для фильтрации данных.
- Для оптимизации работы с данными в датафрейме были произведены следующие изменения типов данных:
    - `year_of_release`: тип данных изменен с `float64` на `int16`;
    - `na_sales`, `other_sales`, `critic_score`: тип данных изменен с `float64` на `float32`;
    - `eu_sales`, `jp_sales`, `user_score`: после замены строковых значений на пропуски тип данных изменен с `object` на  `float32`.
- В процессе обработки пропусков и дубликатов было выполнено следующее:
    - `eu_sales`, `jp_sales`: пропуски по продажам в Европе и Японии заполнены средними значениями по платформе и году выхода  игры;
    - `genre`: исключены дубликаты, связанные с написанием жанра в разных регистрах, путем приведения всех названий к верхнему  регистру;
    - `name`: удалены лишние пробелы в начале и конце названия игры, наименования приведены к нижнему регистру.
    - после нормализации данных с текстовыми значениями были выявлены и удалены 235 явных дубликатов.
- Всего в процессе подготовки данных были удалены 512 строк, что составляет 3.02% от исходного количества.

---

## 3. Фильтрация данных

Коллеги хотят изучить историю продаж игр в начале XXI века, и их интересует период с 2000 по 2013 год включительно. Отберем данные по этому показателю.

In [145]:
df_actual = df[(df['year_of_release'] >= 2000) & (df['year_of_release'] <= 2013)].copy()

In [160]:
df_actual.info()
print(f"\nНаименьший год выпуска в срезе данных: {df_actual['year_of_release'].min()}, \
наибольший год выпуска в срезе данных: {df_actual['year_of_release'].max()}")

<class 'pandas.core.frame.DataFrame'>
Int64Index: 12781 entries, 8460 to 9260
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   name                   12781 non-null  object 
 1   platform               12781 non-null  object 
 2   year_of_release        12781 non-null  int16  
 3   genre                  12781 non-null  object 
 4   na_sales               12781 non-null  float32
 5   eu_sales               12781 non-null  float64
 6   jp_sales               12781 non-null  float64
 7   other_sales            12781 non-null  float32
 8   critic_score           7169 non-null   float32
 9   user_score             6483 non-null   float32
 10  rating                 8723 non-null   object 
 11  category_user_score    6483 non-null   object 
 12  category_critic_score  7169 non-null   object 
dtypes: float32(4), float64(2), int16(1), object(6)
memory usage: 1.1+ MB

Наименьший год выпуска в срезе дан

---

## 4. Категоризация данных
    
Проведем категоризацию данных:
- Разделим все игры по оценкам пользователей и выделим такие категории: высокая оценка (от 8 до 10 включительно), средняя оценка (от 3 до 8, не включая правую границу интервала) и низкая оценка (от 0 до 3, не включая правую границу интервала).

In [147]:
def categorize_user(value):
    if 8 <= value <= 10:
        return 'высокая оценка'
    elif 3 <= value < 8:
        return 'средняя оценка'
    elif 0 <= value < 3:
        return 'низкая оценка'

In [148]:
df_actual['category_user_score'] = df_actual['user_score'].apply(categorize_user)

- Разделим все игры по оценкам критиков и выделите такие категории: высокая оценка (от 80 до 100 включительно), средняя оценка (от 30 до 80, не включая правую границу интервала) и низкая оценка (от 0 до 30, не включая правую границу интервала).

In [149]:
def categorize_critic(value):
    if 80 <= value <= 100:
        return 'высокая оценка'
    elif 30 <= value < 80:
        return 'средняя оценка'
    elif 0 <= value < 30:
        return 'низкая оценка'

In [150]:
df_actual['category_critic_score'] = df_actual['critic_score'].apply(categorize_critic)

- Сгруппируем данные по выделенным категориям и посчитаем количество игр в каждой категории.

In [151]:
# Сгруппируем данные по категориям оценок пользователей
group_user = df_actual.groupby('category_user_score')['name'].count()
group_user

category_user_score
высокая оценка    2286
низкая оценка      116
средняя оценка    4081
Name: name, dtype: int64

In [152]:
# Сгруппируем данные по категориям оценок критиков
group_critic = df_actual.groupby('category_critic_score')['name'].count()
group_critic

category_critic_score
высокая оценка    1692
низкая оценка       55
средняя оценка    5422
Name: name, dtype: int64

In [153]:
# Сводная группировка по категориям оценок пользователей и критиков
grouped = df_actual.groupby(['category_user_score', 'category_critic_score'])['name'].count()
grouped

category_user_score  category_critic_score
высокая оценка       высокая оценка           1017
                     низкая оценка               1
                     средняя оценка           1167
низкая оценка        высокая оценка              1
                     низкая оценка              17
                     средняя оценка             77
средняя оценка       высокая оценка            641
                     низкая оценка              30
                     средняя оценка           3157
Name: name, dtype: int64

- Выделим топ-7 платформ по количеству игр, выпущенных за весь актуальный период.

In [155]:
df_actual['platform'].value_counts().head(7)

PS2     2127
DS      2120
Wii     1275
PSP     1180
X360    1121
PS3     1087
GBA      811
Name: platform, dtype: int64

---

## 5. Итоговый вывод


Изучены данные о продажах игр за период с 2000 по 2013 год включительно. Для этого выделен срез данных `df_actual`.

Была проведена категоризация игр по оценкам пользователей и выделены категории: `высокая оценка` (от 8 до 10 включительно), `средняя оценка` (от 3 до 8, не включая правую границу интервала) и `низкая оценка` (от 0 до 3, не включая правую границу интервала). Для сохранения вычисленной категории в срез данных `df_actual` добавлен столбец `category_user_score`. Наибольшее количество игр — 4081 — получило `среднюю оценку` пользователей, `высокая оценка` выставлена 2286 играм, `низкая` — 116 играм.

Также была проведена категоризация игр по оценкам критиков и выделены категории: `высокая оценка` (от 80 до 100 включительно), `средняя оценка` (от 30 до 80, не включая правую границу интервала) и `низкая оценка` (от 0 до 30, не включая правую границу интервала). Для сохранения вычисленной категории в срез данных `df_actual` добавлен столбец `category_critic_score`. Наибольшее количество игр — 5422 — получило `среднюю оценку` пользователей, `высокая оценка` выставлена 1692 играм, `низкая` — 55 играм.

Выполнен анализ платформ по количеству игр, выпущенных за за период с 2000 по 2013 год. Топ-7 платформ по популярности (по убыванию): PS2, DS, Wii, PSP, X360, PS3, GBA.